# Day 1 · ContactCanvas: protein–ligand interactions

Use this notebook after the PyMOL activity to examine **molecular recognition in an experimental holo structure**. Provide the protein and its bound ligand as separate files while retaining their original shared coordinates, or use the prepared 1IEP example. Upload the protein as PDB and the ligand as `.pdb`, `.sdf`, `.mol2`, or `.mol`.

The notebook produces an interactive 2D interaction map, a compact contact table, a 3D view of the bound ligand in the complete protein, and downloadable HTML and CSV files. A ligand PDB exported from PyMOL is accepted, although SDF or MOL2 is preferable because PDB may not preserve ligand bond orders and formal charges. Interaction assignments depend on protonation and structure quality.

**ContactCanvas does not perform docking, generate poses or score compounds.** It analyses the bound coordinates already present in the holo structure. Compare its assignments with your PyMOL measurements and use the contacts to propose donor, acceptor, aromatic, hydrophobic and charged features for a provisional structure-based pharmacophore.

**Practical design:** Yu Zhu and Taufiq Rahman · Department of Pharmacology, University of Cambridge. Found an error? Contact Yu Zhu at [yz876@cam.ac.uk](mailto:yz876@cam.ac.uk).

In [ ]:
# @title Step 1. Install tools { display-mode: "form" }
%pip -q install 'prolif==2.0.3' 'MDAnalysis>=2.7,<3' 'rdkit>=2024.9' 'py3Dmol>=2.0' 'ipywidgets>=8,<9'

In [ ]:
# @title Step 2. Choose structures { display-mode: "form" }
from pathlib import Path
import json, zipfile, urllib.request
import MDAnalysis as mda
import prolif as plf
from rdkit import Chem
from IPython.display import display, Markdown, clear_output
import py3Dmol
import ipywidgets as widgets

WORK = Path('/content/contactcanvas'); WORK.mkdir(exist_ok=True)
source = 'Prepared 1IEP example' # @param ['Prepared 1IEP example', 'Upload']



if source == 'Prepared 1IEP example':
    protein_file = WORK/'1IEP_chainA_protein.pdb'
    ligand_file = WORK/'1IEP_STI_A201_bound.sdf'
    base = 'https://raw.githubusercontent.com/JudeYu99/molecular-discovery-practical-students/main/docs/examples/'
    urllib.request.urlretrieve(base + protein_file.name, protein_file)
    urllib.request.urlretrieve(base + ligand_file.name, ligand_file)
else:
    from google.colab import files
    print('Upload the protein .pdb file')
    up = files.upload()
    if len(up) != 1 or not next(iter(up)).lower().endswith('.pdb'):
        raise ValueError('Upload exactly one protein .pdb file')
    protein_file = WORK/'protein.pdb'; protein_file.write_bytes(next(iter(up.values())))
    print('Upload the bound ligand .pdb, .sdf, .mol2, or .mol file')
    up = files.upload()
    if len(up) != 1:
        raise ValueError('Upload exactly one bound ligand file')
    filename, data = next(iter(up.items()))
    suffix = Path(filename).suffix.lower()
    if suffix not in ('.pdb', '.sdf', '.mol2', '.mol'):
        raise ValueError('The bound ligand must be .pdb, .sdf, .mol2, or .mol')
    ligand_file = WORK/('ligand' + suffix)
    ligand_file.write_bytes(data)

if not protein_file.exists() or not ligand_file.exists():
    raise FileNotFoundError('Example files unavailable. Choose Upload and provide your files.')
print('Protein:', protein_file.name, '| ligand:', ligand_file.name)

---

## 1 · Read and check the bound pair

The notebook automatically uses the first protein model and the first valid ligand structure. If an uploaded file contains more than one, the notebook reports this choice. PDB, MOL2 and MOL files normally contain one ligand structure in this workflow.

The protein and ligand must come from the same holo structure and remain in the same coordinate frame. A ligand PDB is convenient to export from PyMOL, but it can lose bond-order and charge information. Use SDF or MOL2 when those files are available. ProLIF uses explicit hydrogens to classify hydrogen bonds. If the protein lacks explicit hydrogens, the notebook continues with implicit hydrogen handling and labels the limitation. For reliable hydrogen-bond assignments, use a prepared protein with explicit hydrogens. The notebook reports a minimum heavy-atom distance as a quick coordinate check.

In [ ]:
# @title Step 3. Check alignment { display-mode: "form" }
import numpy as np
from scipy.spatial.distance import cdist

u = mda.Universe(str(protein_file))
if len(u.trajectory) > 1:
    print(f'Note: the protein contains {len(u.trajectory)} models. The first model will be used.')
u.trajectory[0]
if not any(a.element == 'H' for a in u.atoms):
    print('Note: protein has no explicit hydrogens. Continuing with implicit hydrogen handling. H-bond assignments may be incomplete.')
protein_mol = plf.Molecule.from_mda(u) if any(a.element == 'H' for a in u.atoms) else plf.Molecule.from_mda(u, inferrer=None, implicit_hydrogens=True)
if ligand_file.suffix.lower() == '.sdf':
    ligand_structures = [m for m in Chem.SDMolSupplier(str(ligand_file), removeHs=False) if m is not None]
elif ligand_file.suffix.lower() == '.mol2':
    ligand_structures = [Chem.MolFromMol2File(str(ligand_file), removeHs=False)]
elif ligand_file.suffix.lower() == '.pdb':
    ligand_structures = [Chem.MolFromPDBFile(str(ligand_file), removeHs=False)]
    print('Note: ligand PDB accepted. Check the 2D structure because PDB may not preserve bond orders or formal charges.')
else:
    ligand_structures = [Chem.MolFromMolFile(str(ligand_file), removeHs=False)]
ligand_structures = [m for m in ligand_structures if m is not None and m.GetNumConformers() == 1 and m.GetConformer().Is3D()]
if not ligand_structures:
    raise ValueError('No valid 3D ligand structure found. Upload a ligand file exported from the same holo structure as the protein.')
if len(ligand_structures) > 1:
    print(f'Note: the ligand file contains {len(ligand_structures)} valid structures. The first structure will be used.')
selected_ligand = ligand_structures[0]
ligand_mol = plf.Molecule.from_rdkit(selected_ligand)
if not any(a.GetSymbol() == 'H' for a in selected_ligand.GetAtoms()):
    print('Warning: ligand has no explicit hydrogens. H-bond assignments may be incomplete.')
protein_heavy = np.array([a.position for a in u.atoms if a.element != 'H'])
ligand_heavy = np.array([list(selected_ligand.GetConformer().GetAtomPosition(a.GetIdx()))
                         for a in selected_ligand.GetAtoms() if a.GetSymbol() != 'H'])
closest = cdist(protein_heavy, ligand_heavy).min()
print(f'Ligand structure ready. Closest heavy-atom separation: {closest:.2f} Å')
if closest > 8: print('Warning: the protein and ligand may not be in the same coordinate frame.')
if closest < 1: print('Warning: severe overlap. Check the uploaded files.')

---

## 2 · Draw and inspect the interactions

Drag residue labels to tidy the 2D diagram. Hover over edges for distances and click legend entries to hide interaction classes. In the 3D view, change the protein or ligand representation with the menus. The default clean cartoon highlights complete protein residues within 5 Å of the bound ligand as hoverable sticks. Choose **Lines** to inspect atoms throughout the protein. Use **Save current view as PNG** after rotating or zooming.

Compare the diagram with the contacts you identified in PyMOL. Ask which ligand atoms could act as donors, acceptors, aromatic groups, hydrophobic groups or charged features. The CSV records the residue interaction summary. The HTML keeps the interactive 2D diagram.

In [ ]:
# @title Step 4. Draw 2D contacts { display-mode: "form" }
fp = plf.Fingerprint(count=True)
fp.run_from_iterable([ligand_mol], protein_mol)
if not fp.ifp.get(0):
    raise ValueError('No interactions detected. Check the shared coordinates, protonation and bound ligand chemistry.')
from prolif.plotting.network import LigNetwork
view = LigNetwork.from_fingerprint(fp, ligand_mol, kind='frame', frame=0, display_all=False)
display(view.display(height='650px', fontsize=18))
view.save(WORK/'interactions.html')

In [ ]:
# @title Step 5. Contact table { display-mode: "form" }
rows = []
for (lig_res, prot_res), interactions in fp.ifp[0].items():
    for kind, occurrences in interactions.items():
        distances = [entry.get('distance') for entry in occurrences if entry.get('distance') is not None]
        rows.append({'residue': str(prot_res), 'interaction': kind,
                     'nearest_A': min(distances) if distances else None, 'occurrences': len(occurrences)})
import pandas as pd
summary = pd.DataFrame(rows).sort_values(['residue','interaction'])
summary.to_csv(WORK/'interaction_summary.csv', index=False)
display(summary.round({'nearest_A': 2}))

In [ ]:
# @title Step 6. Inspect in 3D { display-mode: "form" }
# Show the selected bound ligand within the complete protein structure.
# The protein and ligand are separate models in their original shared coordinates.
protein_pdb = Chem.MolToPDBBlock(Chem.RemoveAllHs(protein_mol), flavor=0x20 | 0x10)

lig_xyz = np.array([list(selected_ligand.GetConformer().GetAtomPosition(a.GetIdx()))
                    for a in selected_ligand.GetAtoms() if a.GetSymbol() != 'H'])
pocket = set()
for line in protein_pdb.splitlines():
    if not line.startswith('ATOM  ') or line[76:78].strip().upper() == 'H':
        continue
    xyz = np.array([float(line[i:i+8]) for i in (30, 38, 46)])
    if np.linalg.norm(lig_xyz - xyz, axis=1).min() <= 5.0:
        pocket.add((line[21], int(line[22:26])))

protein_menu = widgets.Dropdown(options=['Cartoon + pocket', 'Cartoon', 'Sticks', 'Ball and stick', 'Lines'],
                                value='Cartoon + pocket', description='Protein:',
                                style={'description_width': 'initial'}, layout=widgets.Layout(width='280px'))
ligand_menu = widgets.Dropdown(options=['Sticks', 'Ball and stick', 'Lines'], value='Sticks',
                               description='Ligand:', style={'description_width': 'initial'},
                               layout=widgets.Layout(width='240px'))
three_d_output = widgets.Output()

def add_snapshot_button(view, filename):
    view.startjs += f'''var canvasBox = document.getElementById('3dmolviewer_UNIQUEID');
var saveButton = document.createElement('button');
saveButton.textContent = 'Save current view as PNG';
saveButton.style.cssText = 'display:block;margin:8px 0;padding:9px 14px;background:#145a82;color:white;border:0;border-radius:6px;cursor:pointer;font-weight:600';
saveButton.onclick = function() {{
  viewer_UNIQUEID.render();
  var imageLink = document.createElement('a');
  imageLink.href = viewer_UNIQUEID.pngURI();
  imageLink.download = {json.dumps(filename)};
  document.body.appendChild(imageLink);
  imageLink.click();
  imageLink.remove();
}};
canvasBox.parentNode.insertBefore(saveButton, canvasBox);
'''

def show_complex(change=None):
    with three_d_output:
        clear_output(wait=True)
        three_d = py3Dmol.view(width=850, height=550)
        three_d.addModel(protein_pdb, 'pdb')
        three_d.addModel(Chem.MolToMolBlock(selected_ligand), 'sdf')
        protein_choice = protein_menu.value
        if protein_choice in ('Cartoon + pocket', 'Cartoon'):
            three_d.setStyle({'model': 0}, {'cartoon': {'color': '#aeb9c8'}})
            if protein_choice == 'Cartoon + pocket':
                for chain, resi in sorted(pocket):
                    three_d.addStyle({'model': 0, 'chain': chain, 'resi': resi},
                                     {'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.15}})
        elif protein_choice == 'Sticks':
            three_d.setStyle({'model': 0}, {'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.14}})
        elif protein_choice == 'Ball and stick':
            three_d.setStyle({'model': 0}, {'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.11},
                                            'sphere': {'colorscheme': 'Jmol', 'radius': 0.24}})
        elif protein_choice == 'Lines':
            three_d.setStyle({'model': 0}, {'line': {'colorscheme': 'Jmol', 'linewidth': 1.5}})
        ligand_choice = ligand_menu.value
        if ligand_choice == 'Sticks':
            ligand_style = {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.2}}
        elif ligand_choice == 'Ball and stick':
            ligand_style = {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.12},
                            'sphere': {'colorscheme': 'cyanCarbon', 'radius': 0.27}}
        else:
            ligand_style = {'line': {'colorscheme': 'cyanCarbon', 'linewidth': 2}}
        three_d.setStyle({'model': 1}, ligand_style)
        three_d.startjs += '''viewer_UNIQUEID.getModel(0).setHoverable(
  {}, true,
  function(atom, viewer) {
    if (!atom.residueLabel) {
      var chain = atom.chain || '(blank)';
      var insertion = atom.icode || '';
      atom.residueLabel = viewer.addLabel(
        atom.resn + ' ' + atom.resi + insertion + ' · chain ' + chain + ' · atom ' + atom.atom,
        {position: atom, backgroundColor: 'white', fontColor: '#172b3a',
         borderColor: '#2878b5', borderThickness: 1, fontSize: 14, inFront: true});
    }
  },
  function(atom, viewer) {
    if (atom.residueLabel) {
      viewer.removeLabel(atom.residueLabel);
      delete atom.residueLabel;
    }
  });
'''
        three_d.zoomTo({'model': 1})
        three_d.zoom(0.8)
        add_snapshot_button(three_d, 'contactcanvas_bound_ligand.png')
        three_d.show()

protein_menu.observe(show_complex, names='value')
ligand_menu.observe(show_complex, names='value')
display(widgets.VBox([widgets.HBox([protein_menu, ligand_menu]), three_d_output]))
show_complex()
print(f'Bound ligand and full protein shown. {len(pocket)} nearby residues found. Hover over pocket atoms, or choose Lines for atom-level inspection across the protein.')

In [ ]:
# @title Step 7. Download results { display-mode: "form" }
bundle = WORK/'contactcanvas_results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(WORK/'interactions.html', 'interactions.html')
    z.write(WORK/'interaction_summary.csv', 'interaction_summary.csv')
print('Ready:', bundle)
from google.colab import files
files.download(str(bundle))

### Interpretation

An interaction diagram is a geometric classification, not proof of binding or energetic importance. Check residue protonation, ligand bond orders, structural waters, metals and steric clashes. If the ligand came from a PDB export, inspect its inferred chemistry carefully. Use the diagram together with the 3D structure to support a provisional structure-based pharmacophore and state any uncertainty.

The visual network comes from **ProLIF**. ContactCanvas is our teaching workflow and reporting layer around it.

### References

1. Bouysset C, Fiorucci S. ProLIF: a library to encode molecular interactions as fingerprints. *J Cheminform.* 2021, 13:72. [doi:10.1186/s13321-021-00548-6](https://doi.org/10.1186/s13321-021-00548-6).
2. Rego N, Koes D. 3Dmol.js: molecular visualization with WebGL. *Bioinformatics.* 2015, 31:1322–1324. [doi:10.1093/bioinformatics/btu829](https://doi.org/10.1093/bioinformatics/btu829).
3. Michaud-Agrawal N, Denning EJ, Woolf TB, Beckstein O. MDAnalysis: a toolkit for the analysis of molecular dynamics simulations. *J Comput Chem.* 2011, 32:2319–2327. [doi:10.1002/jcc.21787](https://doi.org/10.1002/jcc.21787).
4. RDKit contributors. *RDKit: Open-source cheminformatics* [software]. [doi:10.5281/zenodo.591637](https://doi.org/10.5281/zenodo.591637).